# 01 - Data Import and Extraction

## Stage 2 Objective

Load legal document files from `data/raw/pdfs`, extract text from PDF, TXT, and DOCX files, preview the results, calculate `word_count` and `char_count`, and save the extracted records to `data/processed/extracted_text.csv`.

## Extraction Strategy

- PDF files are extracted page by page with PyMuPDF first.
- If PyMuPDF fails or returns no page text, the extractor falls back to pdfplumber.
- TXT and DOCX files are extracted as single document-level records.
- Errors are captured as rows so one bad file does not stop the notebook.

## 1. Configure Paths and Imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_loader import (
    SUPPORTED_EXTENSIONS,
    extract_documents_from_directory,
    get_extracted_text_columns,
)

RAW_DOC_DIR = PROJECT_ROOT / 'data' / 'raw' / 'pdfs'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_PATH = PROCESSED_DIR / 'extracted_text.csv'

RAW_DOC_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw document directory: {RAW_DOC_DIR}')
print(f'Supported extensions: {sorted(SUPPORTED_EXTENSIONS)}')

## 2. Discover Input Files

Place small sample `.pdf`, `.txt`, or `.docx` files in `data/raw/pdfs` before running extraction. The folder name is kept from the project structure, but the loader supports all three file types.

In [ ]:
input_files = sorted(
    path
    for path in RAW_DOC_DIR.rglob('*')
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

print(f'Found {len(input_files)} supported file(s).')
for path in input_files[:20]:
    print(f'- {path.relative_to(PROJECT_ROOT)}')

if not input_files:
    print('No files found yet. The notebook will still write an empty output CSV with the expected columns.')

## 3. Extract Text

The reusable extraction logic lives in `src/document_loader.py`. PDF output is page-level; TXT and DOCX output is document-level.

In [ ]:
records = extract_documents_from_directory(RAW_DOC_DIR)
columns = get_extracted_text_columns()
df = pd.DataFrame(records, columns=columns)

if not df.empty:
    df = df.sort_values(['source_path', 'page_number'], na_position='last').reset_index(drop=True)

print(f'Extracted {len(df)} row(s).')
df.head()

## 4. Preview Results and Counts

In [ ]:
preview_df = df.copy()
if not preview_df.empty:
    preview_df['text_preview'] = preview_df['text'].fillna('').str.replace(r'\s+', ' ', regex=True).str.slice(0, 250)
    display_columns = [
        'document_id',
        'source_path',
        'page_number',
        'word_count',
        'char_count',
        'extraction_method',
        'error',
        'text_preview',
    ]
    display(preview_df[display_columns].head(10))
else:
    display(pd.DataFrame(columns=columns + ['text_preview']))

## 5. Check Extraction Summary

In [ ]:
if df.empty:
    print('No extracted text rows to summarize.')
else:
    summary = {
        'documents': df['document_id'].nunique(),
        'rows': len(df),
        'total_words': int(df['word_count'].sum()),
        'total_characters': int(df['char_count'].sum()),
        'rows_with_errors': int((df['error'].fillna('') != '').sum()),
    }
    display(pd.DataFrame([summary]))
    display(df.groupby('extraction_method', dropna=False).size().reset_index(name='row_count'))

## 6. Save Extracted Text

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(df)} row(s) to {OUTPUT_PATH.relative_to(PROJECT_ROOT)}')